In [ ]:
pip install scikit-rf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 627.1/627.1 kB 10.7 MB/s eta 0:00:00


In [ ]:
import skrf as rf
import matplotlib.pyplot as plt
import os

# Obtener una lista de todos los archivos .s2p en el directorio actual
s2p_files = [f for f in os.listdir('.') if f.endswith('.s2p')]

if not s2p_files:
    print("No se encontraron archivos .s2p en el directorio actual.")
else:
    print(f"Se encontraron los siguientes archivos .s2p: {s2p_files}")

    # Lista para almacenar todas las redes cargadas
    networks = []

    for filename in s2p_files:
        try:
            print(f"\nCargando archivo: {filename}")
            # Cargar el archivo S2P
            network = rf.Network(filename)
            networks.append(network)

            # Mostrar información básica del archivo
            #print("Información del archivo S2P:")
            #print(network)

            # Acceder a los datos de la red (por ejemplo, S-parámetros en dB y ángulo)
            #print("\nDatos de la primera frecuencia (S-parámetros):")
            #print(network.s[0]) # Los S-parámetros complejos
            #print(network.s_db[0]) # Los S-parámetros en dB
            #print(network.s_deg[0]) # Los S-parámetros en grados

            #print(f"\nFrecuencia de la primera fila: {network.f[0]} Hz")

            # También puedes acceder a S11, S21, etc. directamente
           # print(f"\nS11 (primera frecuencia, en dB): {network.s11.s_db[0][0]} dB")
            #print(f"S21 (primera frecuencia, en dB): {network.s21.s_db[0][0]} dB")

            # Graficar S21 en dB como ejemplo (puedes ajustar esto según lo necesites)
            # plt.figure() # Crea una nueva figura para cada archivo
            # network.plot_s_db(m=1, n=0, label=f'S21 (dB) - {filename}') # S21 (m=1, n=0)
            # plt.title(f'Ejemplo de S21 en dB para {filename}')
            # plt.xlabel('Frecuencia (Hz)')
            # plt.ylabel('S21 (dB)')
            # plt.grid(True)
            # plt.show()

        except Exception as e:
            print(f"Ocurrió un error al procesar el archivo {filename}: {e}")

    print(f"\nSe cargaron {len(networks)} redes S2P en la lista 'networks'.")
    # Ahora 'networks' contiene todos los objetos Network cargados.

Se encontraron los siguientes archivos .s2p: ['BFP760_VCE_2.0V_IC_50mA.s2p', 'BFP760_w_noise_VCE_3.0V_IC_4.0mA.s2p', 'BFP760_VCE_4.0V_IC_28mA.s2p', 'BFP760_w_noise_VCE_2.0V_IC_10mA.s2p', 'BFP760_VCE_1.5V_IC_60mA.s2p', 'BFP760_w_noise_VCE_4.0V_IC_25mA.s2p', 'BFP760_w_noise_VCE_4.0V_IC_2.0mA.s2p', 'BFP760_VCE_1.0V_IC_55mA.s2p', 'BFP760_VCE_3.5V_IC_28mA.s2p', 'BFP760_VCE_4.0V_IC_65mA.s2p', 'BFP760_VCE_1.5V_IC_5.0mA.s2p', 'BFP760_w_noise_VCE_3.5V_IC_20mA.s2p', 'BFP760_VCE_4.0V_IC_5.0mA.s2p', 'BFP760_w_noise_VCE_2.5V_IC_30mA.s2p', 'BFP760_w_noise_VCE_2.0V_IC_4.0mA.s2p', 'BFP760_VCE_3.5V_IC_55mA.s2p', 'BFP760_VCE_2.5V_IC_60mA.s2p', 'BFP760_w_noise_VCE_2.5V_IC_6.0mA.s2p', 'BFP760_w_noise_VCE_2.0V_IC_40mA.s2p', 'BFP760_VCE_2.5V_IC_65mA.s2p', 'BFP760_w_noise_VCE_3.5V_IC_6.0mA.s2p', 'BFP760_VCE_2.0V_IC_70mA.s2p', 'BFP760_w_noise_VCE_3.0V_IC_2.0mA.s2p', 'BFP760_VCE_2.5V_IC_45mA.s2p', 'BFP760_VCE_1.0V_IC_50mA.s2p', 'BFP760_w_noise_VCE_1.5V_IC_10mA.s2p', 'BFP760_w_noise_VCE_3.0V_IC_6.0mA.s2p', 'BFP

In [ ]:
import numpy as np
import pandas as pd
import re # Importar la librería de expresiones regulares

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

target_frequency = 2.2e9 # 2.2 GHz
extracted_s_parameters = []

# Initialize networks and s2p_files to prevent NameError if previous cell failed to populate them
# (e.g., no .s2p files were found) or if it hasn't been run yet.
if 'networks' not in globals():
    networks = []
if 's2p_files' not in globals():
    s2p_files = []

print(f"Extrayendo parámetros S a {target_frequency/1e9} GHz para cada archivo...")

for i, network in enumerate(networks):
    try:
        # Encontrar el índice de la frecuencia más cercana a la frecuencia objetivo
        freq_index = np.argmin(np.abs(network.f - target_frequency))

        # Obtener los S-parámetros complejos en ese índice
        s_params_at_freq = network.s[freq_index]
        s_db_at_freq = network.s_db[freq_index]
        s_deg_at_freq = network.s_deg[freq_index]

        # Extraer VCE e IC del nombre del archivo
        filename = s2p_files[i]
        vce_match = re.search(r'VCE_([0-9.]+?)V', filename)
        ic_match = re.search(r'IC_([0-9.]+?)mA', filename)

        vce = float(vce_match.group(1)) if vce_match else None
        ic = float(ic_match.group(1)) if ic_match else None

        extracted_s_parameters.append({
            'filename': filename,
            'network_name': network.name,
            'frequency_ghz': network.f[freq_index]/1e9,
            'VCE': vce,
            'IC': ic,
            's11_mag': np.abs(s_params_at_freq[0,0]),
            's21_mag': np.abs(s_params_at_freq[1,0]),
            's12_mag': np.abs(s_params_at_freq[0,1]),
            's22_mag': np.abs(s_params_at_freq[1,1]),
            's11_deg': s_deg_at_freq[0,0],
            's21_deg': s_deg_at_freq[1,0],
            's12_deg': s_deg_at_freq[0,1],
            's22_deg': s_deg_at_freq[1,1]
        })

    except Exception as e:
        print(f"Error al procesar el archivo {s2p_files[i]} para la frecuencia {target_frequency}: {e}")

print("\n--- Resumen de parámetros S extraídos ---")
df_s_parameters = pd.DataFrame(extracted_s_parameters)
display(df_s_parameters)

Extrayendo parámetros S a 2.2 GHz para cada archivo...

--- Resumen de parámetros S extraídos ---


,filename,network_name,frequency_ghz,VCE,IC,s11_mag,s21_mag,s12_mag,s22_mag,s11_deg,s21_deg,s12_deg,s22_deg
0,BFP760_VCE_2.0V_IC_50mA.s2p,BFP760_VCE_2.0V_IC_50mA,2.2,2.0,50.0,0.6744,10.485,0.0384,0.2792,171.7,72.5,37.7,-157.8
1,BFP760_w_noise_VCE_3.0V_IC_4.0mA.s2p,BFP760_w_noise_VCE_3.0V_IC_4.0mA,2.2,3.0,4.0,0.7214,7.122,0.0992,0.5460,-127.9,93.9,18.0,-85.0
2,BFP760_VCE_4.0V_IC_28mA.s2p,BFP760_VCE_4.0V_IC_28mA,2.2,4.0,28.0,0.6461,11.184,0.0429,0.2851,-179.0,76.3,30.7,-140.8
3,BFP760_w_noise_VCE_2.0V_IC_10mA.s2p,BFP760_w_noise_VCE_2.0V_IC_10mA,2.2,2.0,10.0,0.6636,9.407,0.0669,0.3836,-158.6,82.6,18.1,-119.0
4,BFP760_VCE_1.5V_IC_60mA.s2p,BFP760_VCE_1.5V_IC_60mA,2.2,1.5,60.0,0.7895,5.356,0.0357,0.1920,163.9,68.5,37.0,-166.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,BFP760_w_noise_VCE_2.0V_IC_35mA.s2p,BFP760_w_noise_VCE_2.0V_IC_35mA,2.2,2.0,35.0,0.6601,10.813,0.0415,0.2986,176.2,74.2,33.2,-152.9
150,BFP760_VCE_3.0V_IC_70mA.s2p,BFP760_VCE_3.0V_IC_70mA,2.2,3.0,70.0,0.6913,9.889,0.0351,0.2313,168.5,71.6,41.9,-153.8
151,BFP760_VCE_3.5V_IC_22mA.s2p,BFP760_VCE_3.5V_IC_22mA,2.2,3.5,22.0,0.6482,10.919,0.0466,0.3025,-175.2,77.4,27.4,-136.5
152,BFP760_VCE_3.5V_IC_50mA.s2p,BFP760_VCE_3.5V_IC_50mA,2.2,3.5,50.0,0.6555,11.244,0.0378,0.2663,173.5,73.7,38.4,-151.4


In [ ]:
import cmath # Para funciones matemáticas con números complejos (polar, rect)

# Función para convertir magnitud y ángulo (grados) a complejo a+bj
def mag_deg_to_complex(magnitude, angle_deg):
    angle_rad = np.deg2rad(angle_deg)
    real = magnitude * np.cos(angle_rad)
    imag = magnitude * np.sin(angle_rad)
    return complex(real, imag)

# Aplicar la conversión para cada parámetro S
df_s_parameters['s11_complex'] = df_s_parameters.apply(lambda row: mag_deg_to_complex(row['s11_mag'], row['s11_deg']), axis=1)
df_s_parameters['s21_complex'] = df_s_parameters.apply(lambda row: mag_deg_to_complex(row['s21_mag'], row['s21_deg']), axis=1)
df_s_parameters['s12_complex'] = df_s_parameters.apply(lambda row: mag_deg_to_complex(row['s12_mag'], row['s12_deg']), axis=1)
df_s_parameters['s22_complex'] = df_s_parameters.apply(lambda row: mag_deg_to_complex(row['s22_mag'], row['s22_deg']), axis=1)

print("DataFrame con las nuevas columnas de S-parámetros en formato complejo:")
display(df_s_parameters)

DataFrame con las nuevas columnas de S-parámetros en formato complejo:


,filename,network_name,frequency_ghz,VCE,IC,s11_mag,s21_mag,s12_mag,s22_mag,s11_deg,s21_deg,s12_deg,s22_deg,s11_complex,s21_complex,s12_complex,s22_complex
0,BFP760_VCE_2.0V_IC_50mA.s2p,BFP760_VCE_2.0V_IC_50mA,2.2,2.0,50.0,0.6744,10.485,0.0384,0.2792,171.7,72.5,37.7,-157.8,-0.667336+0.097354j,3.152900+ 9.999722j,0.030383+0.023483j,-0.258503-0.105493j
1,BFP760_w_noise_VCE_3.0V_IC_4.0mA.s2p,BFP760_w_noise_VCE_3.0V_IC_4.0mA,2.2,3.0,4.0,0.7214,7.122,0.0992,0.5460,-127.9,93.9,18.0,-85.0,-0.443145-0.569245j,-0.484405+ 7.105507j,0.094345+0.030654j,0.047587-0.543922j
2,BFP760_VCE_4.0V_IC_28mA.s2p,BFP760_VCE_4.0V_IC_28mA,2.2,4.0,28.0,0.6461,11.184,0.0429,0.2851,-179.0,76.3,30.7,-140.8,-0.646002-0.011276j,2.648798+10.865805j,0.036888+0.021902j,-0.220937-0.180192j
3,BFP760_w_noise_VCE_2.0V_IC_10mA.s2p,BFP760_w_noise_VCE_2.0V_IC_10mA,2.2,2.0,10.0,0.6636,9.407,0.0669,0.3836,-158.6,82.6,18.1,-119.0,-0.617849-0.242132j,1.211580+ 9.328651j,0.063590+0.020784j,-0.185973-0.335504j
4,BFP760_VCE_1.5V_IC_60mA.s2p,BFP760_VCE_1.5V_IC_60mA,2.2,1.5,60.0,0.7895,5.356,0.0357,0.1920,163.9,68.5,37.0,-166.0,-0.758535+0.218940j,1.962981+ 4.983316j,0.028511+0.021485j,-0.186297-0.046449j
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,BFP760_w_noise_VCE_2.0V_IC_35mA.s2p,BFP760_w_noise_VCE_2.0V_IC_35mA,2.2,2.0,35.0,0.6601,10.813,0.0415,0.2986,176.2,74.2,33.2,-152.9,-0.658649+0.043747j,2.944166+10.404463j,0.034726+0.022724j,-0.265818-0.136026j
150,BFP760_VCE_3.0V_IC_70mA.s2p,BFP760_VCE_3.0V_IC_70mA,2.2,3.0,70.0,0.6913,9.889,0.0351,0.2313,168.5,71.6,41.9,-153.8,-0.677422+0.137823j,3.121453+ 9.383435j,0.026125+0.023441j,-0.207536-0.102120j
151,BFP760_VCE_3.5V_IC_22mA.s2p,BFP760_VCE_3.5V_IC_22mA,2.2,3.5,22.0,0.6482,10.919,0.0466,0.3025,-175.2,77.4,27.4,-136.5,-0.645927-0.054240j,2.381906+10.656035j,0.041372+0.021445j,-0.219426-0.208227j
152,BFP760_VCE_3.5V_IC_50mA.s2p,BFP760_VCE_3.5V_IC_50mA,2.2,3.5,50.0,0.6555,11.244,0.0378,0.2663,173.5,73.7,38.4,-151.4,-0.651286+0.074205j,3.155816+10.792051j,0.029624+0.023479j,-0.233807-0.127476j


In [ ]:
# Calcular delta = |s11*s22 - s12*s21| usando los valores de magnitud
df_s_parameters['delta'] = ((df_s_parameters['s11_complex'] * df_s_parameters['s22_complex']) - (df_s_parameters['s12_complex'] * df_s_parameters['s21_complex']))
df_s_parameters['delta_modulo'] = np.abs((df_s_parameters['s11_complex'] * df_s_parameters['s22_complex']) - (df_s_parameters['s12_complex'] * df_s_parameters['s21_complex']))

print("DataFrame con la nueva columna 'delta':")
display(df_s_parameters)

DataFrame con la nueva columna 'delta':


,filename,network_name,frequency_ghz,VCE,IC,s11_mag,s21_mag,s12_mag,s22_mag,s11_deg,s21_deg,s12_deg,s22_deg,s11_complex,s21_complex,s12_complex,s22_complex,delta,delta_modulo
0,BFP760_VCE_2.0V_IC_50mA.s2p,BFP760_VCE_2.0V_IC_50mA,2.2,2.0,50.0,0.6744,10.485,0.0384,0.2792,171.7,72.5,37.7,-157.8,-0.667336+0.097354j,3.152900+ 9.999722j,0.030383+0.023483j,-0.258503-0.105493j,0.321804-0.332627j,0.462816
1,BFP760_w_noise_VCE_3.0V_IC_4.0mA.s2p,BFP760_w_noise_VCE_3.0V_IC_4.0mA,2.2,3.0,4.0,0.7214,7.122,0.0992,0.5460,-127.9,93.9,18.0,-85.0,-0.443145-0.569245j,-0.484405+ 7.105507j,0.094345+0.030654j,0.047587-0.543922j,-0.067196-0.441571j,0.446654
2,BFP760_VCE_4.0V_IC_28mA.s2p,BFP760_VCE_4.0V_IC_28mA,2.2,4.0,28.0,0.6461,11.184,0.0429,0.2851,-179.0,76.3,30.7,-140.8,-0.646002-0.011276j,2.648798+10.865805j,0.036888+0.021902j,-0.220937-0.180192j,0.280972-0.339934j,0.441021
3,BFP760_w_noise_VCE_2.0V_IC_10mA.s2p,BFP760_w_noise_VCE_2.0V_IC_10mA,2.2,2.0,10.0,0.6636,9.407,0.0669,0.3836,-158.6,82.6,18.1,-119.0,-0.617849-0.242132j,1.211580+ 9.328651j,0.063590+0.020784j,-0.185973-0.335504j,0.150512-0.366065j,0.395800
4,BFP760_VCE_1.5V_IC_60mA.s2p,BFP760_VCE_1.5V_IC_60mA,2.2,1.5,60.0,0.7895,5.356,0.0357,0.1920,163.9,68.5,37.0,-166.0,-0.758535+0.218940j,1.962981+ 4.983316j,0.028511+0.021485j,-0.186297-0.046449j,0.202581-0.189810j,0.277609
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,BFP760_w_noise_VCE_2.0V_IC_35mA.s2p,BFP760_w_noise_VCE_2.0V_IC_35mA,2.2,2.0,35.0,0.6601,10.813,0.0415,0.2986,176.2,74.2,33.2,-152.9,-0.658649+0.043747j,2.944166+10.404463j,0.034726+0.022724j,-0.265818-0.136026j,0.315223-0.350241j,0.471205
150,BFP760_VCE_3.0V_IC_70mA.s2p,BFP760_VCE_3.0V_IC_70mA,2.2,3.0,70.0,0.6913,9.889,0.0351,0.2313,168.5,71.6,41.9,-153.8,-0.677422+0.137823j,3.121453+ 9.383435j,0.026125+0.023441j,-0.207536-0.102120j,0.293071-0.277740j,0.403770
151,BFP760_VCE_3.5V_IC_22mA.s2p,BFP760_VCE_3.5V_IC_22mA,2.2,3.5,22.0,0.6482,10.919,0.0466,0.3025,-175.2,77.4,27.4,-136.5,-0.645927-0.054240j,2.381906+10.656035j,0.041372+0.021445j,-0.219426-0.208227j,0.260416-0.345543j,0.432685
152,BFP760_VCE_3.5V_IC_50mA.s2p,BFP760_VCE_3.5V_IC_50mA,2.2,3.5,50.0,0.6555,11.244,0.0378,0.2663,173.5,73.7,38.4,-151.4,-0.651286+0.074205j,3.155816+10.792051j,0.029624+0.023479j,-0.233807-0.127476j,0.321639-0.328123j,0.459473


In [ ]:
#Calcular K
df_s_parameters['K'] = ((1 - np.abs(df_s_parameters['s11_complex'])**2 - np.abs(df_s_parameters['s22_complex'])**2 + np.abs(df_s_parameters['delta'])**2) / (2 * np.abs(df_s_parameters['s12_complex'] * df_s_parameters['s21_complex'])))

print("DataFrame con la nueva columna 'K':")
display(df_s_parameters)

DataFrame con la nueva columna 'K':


,filename,network_name,frequency_ghz,VCE,IC,s11_mag,s21_mag,s12_mag,s22_mag,s11_deg,s21_deg,s12_deg,s22_deg,s11_complex,s21_complex,s12_complex,s22_complex,delta,delta_modulo,K
0,BFP760_VCE_2.0V_IC_50mA.s2p,BFP760_VCE_2.0V_IC_50mA,2.2,2.0,50.0,0.6744,10.485,0.0384,0.2792,171.7,72.5,37.7,-157.8,-0.667336+0.097354j,3.152900+ 9.999722j,0.030383+0.023483j,-0.258503-0.105493j,0.321804-0.332627j,0.462816,0.846237
1,BFP760_w_noise_VCE_3.0V_IC_4.0mA.s2p,BFP760_w_noise_VCE_3.0V_IC_4.0mA,2.2,3.0,4.0,0.7214,7.122,0.0992,0.5460,-127.9,93.9,18.0,-85.0,-0.443145-0.569245j,-0.484405+ 7.105507j,0.094345+0.030654j,0.047587-0.543922j,-0.067196-0.441571j,0.446654,0.269614
2,BFP760_VCE_4.0V_IC_28mA.s2p,BFP760_VCE_4.0V_IC_28mA,2.2,4.0,28.0,0.6461,11.184,0.0429,0.2851,-179.0,76.3,30.7,-140.8,-0.646002-0.011276j,2.648798+10.865805j,0.036888+0.021902j,-0.220937-0.180192j,0.280972-0.339934j,0.441021,0.725075
3,BFP760_w_noise_VCE_2.0V_IC_10mA.s2p,BFP760_w_noise_VCE_2.0V_IC_10mA,2.2,2.0,10.0,0.6636,9.407,0.0669,0.3836,-158.6,82.6,18.1,-119.0,-0.617849-0.242132j,1.211580+ 9.328651j,0.063590+0.020784j,-0.185973-0.335504j,0.150512-0.366065j,0.395800,0.452183
4,BFP760_VCE_1.5V_IC_60mA.s2p,BFP760_VCE_1.5V_IC_60mA,2.2,1.5,60.0,0.7895,5.356,0.0357,0.1920,163.9,68.5,37.0,-166.0,-0.758535+0.218940j,1.962981+ 4.983316j,0.028511+0.021485j,-0.186297-0.046449j,0.202581-0.189810j,0.277609,1.090147
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,BFP760_w_noise_VCE_2.0V_IC_35mA.s2p,BFP760_w_noise_VCE_2.0V_IC_35mA,2.2,2.0,35.0,0.6601,10.813,0.0415,0.2986,176.2,74.2,33.2,-152.9,-0.658649+0.043747j,2.944166+10.404463j,0.034726+0.022724j,-0.265818-0.136026j,0.315223-0.350241j,0.471205,0.776776
150,BFP760_VCE_3.0V_IC_70mA.s2p,BFP760_VCE_3.0V_IC_70mA,2.2,3.0,70.0,0.6913,9.889,0.0351,0.2313,168.5,71.6,41.9,-153.8,-0.677422+0.137823j,3.121453+ 9.383435j,0.026125+0.023441j,-0.207536-0.102120j,0.293071-0.277740j,0.403770,0.909864
151,BFP760_VCE_3.5V_IC_22mA.s2p,BFP760_VCE_3.5V_IC_22mA,2.2,3.5,22.0,0.6482,10.919,0.0466,0.3025,-175.2,77.4,27.4,-136.5,-0.645927-0.054240j,2.381906+10.656035j,0.041372+0.021445j,-0.219426-0.208227j,0.260416-0.345543j,0.432685,0.663830
152,BFP760_VCE_3.5V_IC_50mA.s2p,BFP760_VCE_3.5V_IC_50mA,2.2,3.5,50.0,0.6555,11.244,0.0378,0.2663,173.5,73.7,38.4,-151.4,-0.651286+0.074205j,3.155816+10.792051j,0.029624+0.023479j,-0.233807-0.127476j,0.321639-0.328123j,0.459473,0.835860


In [ ]:
filtered_df = df_s_parameters[(df_s_parameters['delta'] < 1) & (df_s_parameters['K'] > 1)]

print("Filas donde delta < 1 y K > 1:")
display(filtered_df)

Filas donde delta < 1 y K > 1:


,filename,network_name,frequency_ghz,VCE,IC,s11_mag,s21_mag,s12_mag,s22_mag,s11_deg,s21_deg,s12_deg,s22_deg,s11_complex,s21_complex,s12_complex,s22_complex,delta,delta_modulo,K
4,BFP760_VCE_1.5V_IC_60mA.s2p,BFP760_VCE_1.5V_IC_60mA,2.2,1.5,60.0,0.7895,5.356,0.0357,0.1920,163.9,68.5,37.0,-166.0,-0.758535+0.218940j,1.962981+4.983316j,0.028511+0.021485j,-0.186297-0.046449j,0.202581-0.189810j,0.277609,1.090147
7,BFP760_VCE_1.0V_IC_55mA.s2p,BFP760_VCE_1.0V_IC_55mA,2.2,1.0,55.0,0.8468,2.617,0.0388,0.2446,162.1,67.2,33.5,172.7,-0.805810+0.260270j,1.014128+2.412516j,0.032355+0.021415j,-0.242617+0.031080j,0.206267-0.187965j,0.279064,1.482069
21,BFP760_VCE_2.0V_IC_70mA.s2p,BFP760_VCE_2.0V_IC_70mA,2.2,2.0,70.0,0.7915,5.262,0.0337,0.1558,163.7,68.9,39.5,-155.5,-0.759686+0.222148j,1.894303+4.909202j,0.026004+0.021436j,-0.141772-0.064609j,0.178029-0.150675j,0.233232,1.138140
24,BFP760_VCE_1.0V_IC_50mA.s2p,BFP760_VCE_1.0V_IC_50mA,2.2,1.0,50.0,0.8110,4.290,0.0394,0.2570,163.2,67.7,32.5,179.1,-0.776386+0.234405j,1.627867+3.969150j,0.033230+0.021170j,-0.256968+0.004037j,0.228492-0.229723j,0.324009,1.127671
30,BFP760_VCE_1.0V_IC_60mA.s2p,BFP760_VCE_1.0V_IC_60mA,2.2,1.0,60.0,0.8655,1.886,0.0379,0.2516,160.7,65.9,35.2,169.0,-0.816860+0.286060j,0.770111+1.721605j,0.030970+0.021847j,-0.246977+0.048008j,0.201774-0.180008j,0.270399,1.823763
62,BFP760_VCE_1.0V_IC_65mA.s2p,BFP760_VCE_1.0V_IC_65mA,2.2,1.0,65.0,0.8756,1.627,0.0373,0.2604,159.3,64.3,37.0,167.9,-0.819075+0.309503j,0.705563+1.466052j,0.029789+0.022448j,-0.254615+0.054585j,0.203546-0.183024j,0.273731,1.981022
107,BFP760_VCE_1.0V_IC_70mA.s2p,BFP760_VCE_1.0V_IC_70mA,2.2,1.0,70.0,0.8826,1.507,0.0369,0.2691,158.1,62.9,38.0,167.7,-0.818908+0.329199j,0.686506+1.341551j,0.029078+0.022718j,-0.262923+0.057326j,0.206953-0.188104j,0.279666,2.039401
109,BFP760_VCE_1.5V_IC_70mA.s2p,BFP760_VCE_1.5V_IC_70mA,2.2,1.5,70.0,0.8478,2.677,0.0346,0.1600,161.4,67.4,38.2,-175.1,-0.803518+0.270414j,1.028759+2.471434j,0.027191+0.021397j,-0.159415-0.013667j,0.156697-0.121339j,0.198185,1.591983
112,BFP760_VCE_1.5V_IC_65mA.s2p,BFP760_VCE_1.5V_IC_65mA,2.2,1.5,65.0,0.8257,3.637,0.0351,0.1650,162.6,68.1,37.2,-169.8,-0.787916+0.246918j,1.356557+3.374540j,0.027958+0.021221j,-0.162392-0.029219j,0.168852-0.140210j,0.219476,1.328402


In [ ]:
# Calcular coeficientes auxiliares B1, B2, C1, C2
df_s_parameters['B1'] = 1 + np.abs(df_s_parameters['s11_complex'])**2 - np.abs((df_s_parameters['s22_complex']))**2 - np.abs(df_s_parameters['delta'])**2
df_s_parameters['B2'] = 1 + np.abs(df_s_parameters['s22_complex'])**2 - np.abs((df_s_parameters['s11_complex']))**2 - np.abs(df_s_parameters['delta'])**2

df_s_parameters['C1'] = (df_s_parameters['s11_complex'] - (df_s_parameters['delta'] * (np.conj(df_s_parameters['s22_complex']))))

df_s_parameters['C2'] = (df_s_parameters['s22_complex'] - (df_s_parameters['delta'] * (np.conj(df_s_parameters['s11_complex']))))

print("DataFrame con las nuevas columnas de coeficientes auxiliares:")
display(df_s_parameters)

DataFrame con las nuevas columnas de coeficientes auxiliares:


,filename,network_name,frequency_ghz,VCE,IC,s11_mag,s21_mag,s12_mag,s22_mag,s11_deg,s21_deg,s12_deg,s22_deg,s11_complex,s21_complex,s12_complex,s22_complex,delta,delta_modulo,K,B1,B2,C1,C2
0,BFP760_VCE_2.0V_IC_50mA.s2p,BFP760_VCE_2.0V_IC_50mA,2.2,2.0,50.0,0.6744,10.485,0.0384,0.2792,171.7,72.5,37.7,-157.8,-0.667336+0.097354j,3.152900+ 9.999722j,0.030383+0.023483j,-0.258503-0.105493j,0.321804-0.332627j,0.462816,0.846237,1.162664,0.408939,-0.619239-0.022579j,-0.011369-0.296138j
1,BFP760_w_noise_VCE_3.0V_IC_4.0mA.s2p,BFP760_w_noise_VCE_3.0V_IC_4.0mA,2.2,3.0,4.0,0.7214,7.122,0.0992,0.5460,-127.9,93.9,18.0,-85.0,-0.443145-0.569245j,-0.484405+ 7.105507j,0.094345+0.030654j,0.047587-0.543922j,-0.067196-0.441571j,0.446654,0.269614,1.022802,0.578198,-0.680128-0.511683j,-0.233553-0.701351j
2,BFP760_VCE_4.0V_IC_28mA.s2p,BFP760_VCE_4.0V_IC_28mA,2.2,4.0,28.0,0.6461,11.184,0.0429,0.2851,-179.0,76.3,30.7,-140.8,-0.646002-0.011276j,2.648798+10.865805j,0.036888+0.021902j,-0.220937-0.180192j,0.280972-0.339934j,0.441021,0.725075,1.141663,0.469337,-0.645178-0.137009j,-0.043262-0.402957j
3,BFP760_w_noise_VCE_2.0V_IC_10mA.s2p,BFP760_w_noise_VCE_2.0V_IC_10mA,2.2,2.0,10.0,0.6636,9.407,0.0669,0.3836,-158.6,82.6,18.1,-119.0,-0.617849-0.242132j,1.211580+ 9.328651j,0.063590+0.020784j,-0.185973-0.335504j,0.150512-0.366065j,0.395800,0.452183,1.136558,0.550126,-0.712674-0.360708j,-0.181616-0.598121j
4,BFP760_VCE_1.5V_IC_60mA.s2p,BFP760_VCE_1.5V_IC_60mA,2.2,1.5,60.0,0.7895,5.356,0.0357,0.1920,163.9,68.5,37.0,-166.0,-0.758535+0.218940j,1.962981+ 4.983316j,0.028511+0.021485j,-0.186297-0.046449j,0.202581-0.189810j,0.277609,1.090147,1.509380,0.336487,-0.729611+0.174169j,0.008925-0.146073j
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,BFP760_w_noise_VCE_2.0V_IC_35mA.s2p,BFP760_w_noise_VCE_2.0V_IC_35mA,2.2,2.0,35.0,0.6601,10.813,0.0415,0.2986,176.2,74.2,33.2,-152.9,-0.658649+0.043747j,2.944166+10.404463j,0.034726+0.022724j,-0.265818-0.136026j,0.315223-0.350241j,0.471205,0.776776,1.124536,0.431396,-0.622499-0.092231j,-0.042874-0.352921j
150,BFP760_VCE_3.0V_IC_70mA.s2p,BFP760_VCE_3.0V_IC_70mA,2.2,3.0,70.0,0.6913,9.889,0.0351,0.2313,168.5,71.6,41.9,-153.8,-0.677422+0.137823j,3.121453+ 9.383435j,0.026125+0.023441j,-0.207536-0.102120j,0.293071-0.277740j,0.403770,0.909864,1.261366,0.412574,-0.644962+0.050254j,0.029276-0.249875j
151,BFP760_VCE_3.5V_IC_22mA.s2p,BFP760_VCE_3.5V_IC_22mA,2.2,3.5,22.0,0.6482,10.919,0.0466,0.3025,-175.2,77.4,27.4,-136.5,-0.645927-0.054240j,2.381906+10.656035j,0.041372+0.021445j,-0.219426-0.208227j,0.260416-0.345543j,0.432685,0.663830,1.141440,0.484126,-0.660736-0.184287j,-0.069958-0.445548j
152,BFP760_VCE_3.5V_IC_50mA.s2p,BFP760_VCE_3.5V_IC_50mA,2.2,3.5,50.0,0.6555,11.244,0.0378,0.2663,173.5,73.7,38.4,-151.4,-0.651286+0.074205j,3.155816+10.792051j,0.029624+0.023479j,-0.233807-0.127476j,0.321639-0.328123j,0.459473,0.835860,1.147649,0.430120,-0.617913-0.043514j,0.000020-0.317310j


In [ ]:
# Calcular coeficientes de reflexion adaptados (matched)
df_s_parameters['ro_s(m)'] = (df_s_parameters['B1'] - (np.sqrt((df_s_parameters['B1']**2) -(4*(np.abs(df_s_parameters['C1']))**2)))) / (2 * (df_s_parameters['C1']))
df_s_parameters['ro_L(m)'] = (df_s_parameters['B2'] - (np.sqrt((df_s_parameters['B2']**2) -(4*(np.abs(df_s_parameters['C2']))**2)))) / (2 * (df_s_parameters['C2']))


print("DataFrame con las nuevas columnas de coeficientes de reflexion adaptados:")
display(df_s_parameters)

DataFrame con las nuevas columnas de coeficientes de reflexion adaptados:


/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


,filename,network_name,frequency_ghz,VCE,IC,s11_mag,s21_mag,s12_mag,s22_mag,s11_deg,s21_deg,s12_deg,s22_deg,s11_complex,s21_complex,s12_complex,s22_complex,delta,delta_modulo,K,B1,B2,C1,C2,ro_s(m),ro_L(m)
0,BFP760_VCE_2.0V_IC_50mA.s2p,BFP760_VCE_2.0V_IC_50mA,2.2,2.0,50.0,0.6744,10.485,0.0384,0.2792,171.7,72.5,37.7,-157.8,-0.667336+0.097354j,3.152900+ 9.999722j,0.030383+0.023483j,-0.258503-0.105493j,0.321804-0.332627j,0.462816,0.846237,1.162664,0.408939,-0.619239-0.022579j,-0.011369-0.296138j,NaN+ NaNj,NaN+ NaNj
1,BFP760_w_noise_VCE_3.0V_IC_4.0mA.s2p,BFP760_w_noise_VCE_3.0V_IC_4.0mA,2.2,3.0,4.0,0.7214,7.122,0.0992,0.5460,-127.9,93.9,18.0,-85.0,-0.443145-0.569245j,-0.484405+ 7.105507j,0.094345+0.030654j,0.047587-0.543922j,-0.067196-0.441571j,0.446654,0.269614,1.022802,0.578198,-0.680128-0.511683j,-0.233553-0.701351j,NaN+ NaNj,NaN+ NaNj
2,BFP760_VCE_4.0V_IC_28mA.s2p,BFP760_VCE_4.0V_IC_28mA,2.2,4.0,28.0,0.6461,11.184,0.0429,0.2851,-179.0,76.3,30.7,-140.8,-0.646002-0.011276j,2.648798+10.865805j,0.036888+0.021902j,-0.220937-0.180192j,0.280972-0.339934j,0.441021,0.725075,1.141663,0.469337,-0.645178-0.137009j,-0.043262-0.402957j,NaN+ NaNj,NaN+ NaNj
3,BFP760_w_noise_VCE_2.0V_IC_10mA.s2p,BFP760_w_noise_VCE_2.0V_IC_10mA,2.2,2.0,10.0,0.6636,9.407,0.0669,0.3836,-158.6,82.6,18.1,-119.0,-0.617849-0.242132j,1.211580+ 9.328651j,0.063590+0.020784j,-0.185973-0.335504j,0.150512-0.366065j,0.395800,0.452183,1.136558,0.550126,-0.712674-0.360708j,-0.181616-0.598121j,NaN+ NaNj,NaN+ NaNj
4,BFP760_VCE_1.5V_IC_60mA.s2p,BFP760_VCE_1.5V_IC_60mA,2.2,1.5,60.0,0.7895,5.356,0.0357,0.1920,163.9,68.5,37.0,-166.0,-0.758535+0.218940j,1.962981+ 4.983316j,0.028511+0.021485j,-0.186297-0.046449j,0.202581-0.189810j,0.277609,1.090147,1.509380,0.336487,-0.729611+0.174169j,0.008925-0.146073j,-0.870981-0.207916j,0.035522+0.581403j
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,BFP760_w_noise_VCE_2.0V_IC_35mA.s2p,BFP760_w_noise_VCE_2.0V_IC_35mA,2.2,2.0,35.0,0.6601,10.813,0.0415,0.2986,176.2,74.2,33.2,-152.9,-0.658649+0.043747j,2.944166+10.404463j,0.034726+0.022724j,-0.265818-0.136026j,0.315223-0.350241j,0.471205,0.776776,1.124536,0.431396,-0.622499-0.092231j,-0.042874-0.352921j,NaN+ NaNj,NaN+ NaNj
150,BFP760_VCE_3.0V_IC_70mA.s2p,BFP760_VCE_3.0V_IC_70mA,2.2,3.0,70.0,0.6913,9.889,0.0351,0.2313,168.5,71.6,41.9,-153.8,-0.677422+0.137823j,3.121453+ 9.383435j,0.026125+0.023441j,-0.207536-0.102120j,0.293071-0.277740j,0.403770,0.909864,1.261366,0.412574,-0.644962+0.050254j,0.029276-0.249875j,NaN+ NaNj,NaN+ NaNj
151,BFP760_VCE_3.5V_IC_22mA.s2p,BFP760_VCE_3.5V_IC_22mA,2.2,3.5,22.0,0.6482,10.919,0.0466,0.3025,-175.2,77.4,27.4,-136.5,-0.645927-0.054240j,2.381906+10.656035j,0.041372+0.021445j,-0.219426-0.208227j,0.260416-0.345543j,0.432685,0.663830,1.141440,0.484126,-0.660736-0.184287j,-0.069958-0.445548j,NaN+ NaNj,NaN+ NaNj
152,BFP760_VCE_3.5V_IC_50mA.s2p,BFP760_VCE_3.5V_IC_50mA,2.2,3.5,50.0,0.6555,11.244,0.0378,0.2663,173.5,73.7,38.4,-151.4,-0.651286+0.074205j,3.155816+10.792051j,0.029624+0.023479j,-0.233807-0.127476j,0.321639-0.328123j,0.459473,0.835860,1.147649,0.430120,-0.617913-0.043514j,0.000020-0.317310j,NaN+ NaNj,NaN+ NaNj


In [ ]:
filtered_df = df_s_parameters[(df_s_parameters['delta'] < 1) & (df_s_parameters['K'] > 1)]

print("Filas donde delta < 1 y K > 1:")
display(filtered_df)

Filas donde delta < 1 y K > 1:


,filename,network_name,frequency_ghz,VCE,IC,s11_mag,s21_mag,s12_mag,s22_mag,s11_deg,s21_deg,s12_deg,s22_deg,s11_complex,s21_complex,s12_complex,s22_complex,delta,delta_modulo,K,B1,B2,C1,C2,ro_s(m),ro_L(m)
4,BFP760_VCE_1.5V_IC_60mA.s2p,BFP760_VCE_1.5V_IC_60mA,2.2,1.5,60.0,0.7895,5.356,0.0357,0.1920,163.9,68.5,37.0,-166.0,-0.758535+0.218940j,1.962981+4.983316j,0.028511+0.021485j,-0.186297-0.046449j,0.202581-0.189810j,0.277609,1.090147,1.509380,0.336487,-0.729611+0.174169j,0.008925-0.146073j,-0.870981-0.207916j,0.035522+0.581403j
7,BFP760_VCE_1.0V_IC_55mA.s2p,BFP760_VCE_1.0V_IC_55mA,2.2,1.0,55.0,0.8468,2.617,0.0388,0.2446,162.1,67.2,33.5,172.7,-0.805810+0.260270j,1.014128+2.412516j,0.032355+0.021415j,-0.242617+0.031080j,0.206267-0.187965j,0.279064,1.482069,1.579364,0.264882,-0.749924+0.221077j,-0.027484-0.066699j,-0.832553-0.245436j,-0.112865+0.273905j
21,BFP760_VCE_2.0V_IC_70mA.s2p,BFP760_VCE_2.0V_IC_70mA,2.2,2.0,70.0,0.7915,5.262,0.0337,0.1558,163.7,68.9,39.5,-155.5,-0.759686+0.222148j,1.894303+4.909202j,0.026004+0.021436j,-0.141772-0.064609j,0.178029-0.150675j,0.233232,1.138140,1.547801,0.343404,-0.744181+0.189284j,0.026946-0.139526j,-0.855111-0.217499j,0.100517+0.520474j
24,BFP760_VCE_1.0V_IC_50mA.s2p,BFP760_VCE_1.0V_IC_50mA,2.2,1.0,50.0,0.8110,4.290,0.0394,0.2570,163.2,67.7,32.5,179.1,-0.776386+0.234405j,1.627867+3.969150j,0.033230+0.021170j,-0.256968+0.004037j,0.228492-0.229723j,0.324009,1.127671,1.486690,0.303346,-0.716743+0.176296j,-0.025722-0.120758j,-0.862050-0.212036j,-0.107278+0.503642j
30,BFP760_VCE_1.0V_IC_60mA.s2p,BFP760_VCE_1.0V_IC_60mA,2.2,1.0,60.0,0.8655,1.886,0.0379,0.2516,160.7,65.9,35.2,169.0,-0.816860+0.286060j,0.770111+1.721605j,0.030970+0.021847j,-0.246977+0.048008j,0.201774-0.180008j,0.270399,1.823763,1.612672,0.241097,-0.758384+0.251289j,-0.030663-0.041314j,-0.828515-0.274527j,-0.133570+0.179967j
62,BFP760_VCE_1.0V_IC_65mA.s2p,BFP760_VCE_1.0V_IC_65mA,2.2,1.0,65.0,0.8756,1.627,0.0373,0.2604,159.3,64.3,37.0,167.9,-0.819075+0.309503j,0.705563+1.466052j,0.029789+0.022448j,-0.254615+0.054585j,0.203546-0.183024j,0.273731,1.981022,1.623939,0.226204,-0.757259+0.274013j,-0.031249-0.032327j,-0.826927-0.299222j,-0.144083+0.149054j
107,BFP760_VCE_1.0V_IC_70mA.s2p,BFP760_VCE_1.0V_IC_70mA,2.2,1.0,70.0,0.8826,1.507,0.0369,0.2691,158.1,62.9,38.0,167.7,-0.818908+0.329199j,0.686506+1.341551j,0.029078+0.022718j,-0.262923+0.057326j,0.206953-0.188104j,0.279666,2.039401,1.628355,0.215219,-0.753712+0.291606j,-0.031524-0.028585j,-0.825519-0.319388j,-0.152695+0.138460j
109,BFP760_VCE_1.5V_IC_70mA.s2p,BFP760_VCE_1.5V_IC_70mA,2.2,1.5,70.0,0.8478,2.677,0.0346,0.1600,161.4,67.4,38.2,-175.1,-0.803518+0.270414j,1.028759+2.471434j,0.027191+0.021397j,-0.159415-0.013667j,0.156697-0.121339j,0.198185,1.591983,1.653888,0.267558,-0.780196+0.248929j,-0.000695-0.068792j,-0.828517-0.264346j,-0.002795+0.276812j
112,BFP760_VCE_1.5V_IC_65mA.s2p,BFP760_VCE_1.5V_IC_65mA,2.2,1.5,65.0,0.8257,3.637,0.0351,0.1650,162.6,68.1,37.2,-169.8,-0.787916+0.246918j,1.356557+3.374540j,0.027958+0.021221j,-0.162392-0.029219j,0.168852-0.140210j,0.219476,1.328402,1.606386,0.297275,-0.764593+0.219215j,0.005269-0.098000j,-0.835782-0.239626j,0.020245+0.376535j


In [ ]:
# Calcular coeficientes ro_in y ro_out
df_s_parameters['ro_in'] = np.conj(df_s_parameters['ro_s(m)'])
df_s_parameters['ro_out'] = np.conj(df_s_parameters['ro_L(m)'])


print("DataFrame con las nuevas columnas de coeficientes ro_in y ro_out:")
display(df_s_parameters)

DataFrame con las nuevas columnas de coeficientes ro_in y ro_out:


,filename,network_name,frequency_ghz,VCE,IC,s11_mag,s21_mag,s12_mag,s22_mag,s11_deg,s21_deg,s12_deg,s22_deg,s11_complex,s21_complex,s12_complex,s22_complex,delta,delta_modulo,K,B1,B2,C1,C2,ro_s(m),ro_L(m),ro_in,ro_out
0,BFP760_VCE_2.0V_IC_50mA.s2p,BFP760_VCE_2.0V_IC_50mA,2.2,2.0,50.0,0.6744,10.485,0.0384,0.2792,171.7,72.5,37.7,-157.8,-0.667336+0.097354j,3.152900+ 9.999722j,0.030383+0.023483j,-0.258503-0.105493j,0.321804-0.332627j,0.462816,0.846237,1.162664,0.408939,-0.619239-0.022579j,-0.011369-0.296138j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj
1,BFP760_w_noise_VCE_3.0V_IC_4.0mA.s2p,BFP760_w_noise_VCE_3.0V_IC_4.0mA,2.2,3.0,4.0,0.7214,7.122,0.0992,0.5460,-127.9,93.9,18.0,-85.0,-0.443145-0.569245j,-0.484405+ 7.105507j,0.094345+0.030654j,0.047587-0.543922j,-0.067196-0.441571j,0.446654,0.269614,1.022802,0.578198,-0.680128-0.511683j,-0.233553-0.701351j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj
2,BFP760_VCE_4.0V_IC_28mA.s2p,BFP760_VCE_4.0V_IC_28mA,2.2,4.0,28.0,0.6461,11.184,0.0429,0.2851,-179.0,76.3,30.7,-140.8,-0.646002-0.011276j,2.648798+10.865805j,0.036888+0.021902j,-0.220937-0.180192j,0.280972-0.339934j,0.441021,0.725075,1.141663,0.469337,-0.645178-0.137009j,-0.043262-0.402957j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj
3,BFP760_w_noise_VCE_2.0V_IC_10mA.s2p,BFP760_w_noise_VCE_2.0V_IC_10mA,2.2,2.0,10.0,0.6636,9.407,0.0669,0.3836,-158.6,82.6,18.1,-119.0,-0.617849-0.242132j,1.211580+ 9.328651j,0.063590+0.020784j,-0.185973-0.335504j,0.150512-0.366065j,0.395800,0.452183,1.136558,0.550126,-0.712674-0.360708j,-0.181616-0.598121j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj
4,BFP760_VCE_1.5V_IC_60mA.s2p,BFP760_VCE_1.5V_IC_60mA,2.2,1.5,60.0,0.7895,5.356,0.0357,0.1920,163.9,68.5,37.0,-166.0,-0.758535+0.218940j,1.962981+ 4.983316j,0.028511+0.021485j,-0.186297-0.046449j,0.202581-0.189810j,0.277609,1.090147,1.509380,0.336487,-0.729611+0.174169j,0.008925-0.146073j,-0.870981-0.207916j,0.035522+0.581403j,-0.870981+0.207916j,0.035522-0.581403j
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,BFP760_w_noise_VCE_2.0V_IC_35mA.s2p,BFP760_w_noise_VCE_2.0V_IC_35mA,2.2,2.0,35.0,0.6601,10.813,0.0415,0.2986,176.2,74.2,33.2,-152.9,-0.658649+0.043747j,2.944166+10.404463j,0.034726+0.022724j,-0.265818-0.136026j,0.315223-0.350241j,0.471205,0.776776,1.124536,0.431396,-0.622499-0.092231j,-0.042874-0.352921j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj
150,BFP760_VCE_3.0V_IC_70mA.s2p,BFP760_VCE_3.0V_IC_70mA,2.2,3.0,70.0,0.6913,9.889,0.0351,0.2313,168.5,71.6,41.9,-153.8,-0.677422+0.137823j,3.121453+ 9.383435j,0.026125+0.023441j,-0.207536-0.102120j,0.293071-0.277740j,0.403770,0.909864,1.261366,0.412574,-0.644962+0.050254j,0.029276-0.249875j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj
151,BFP760_VCE_3.5V_IC_22mA.s2p,BFP760_VCE_3.5V_IC_22mA,2.2,3.5,22.0,0.6482,10.919,0.0466,0.3025,-175.2,77.4,27.4,-136.5,-0.645927-0.054240j,2.381906+10.656035j,0.041372+0.021445j,-0.219426-0.208227j,0.260416-0.345543j,0.432685,0.663830,1.141440,0.484126,-0.660736-0.184287j,-0.069958-0.445548j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj
152,BFP760_VCE_3.5V_IC_50mA.s2p,BFP760_VCE_3.5V_IC_50mA,2.2,3.5,50.0,0.6555,11.244,0.0378,0.2663,173.5,73.7,38.4,-151.4,-0.651286+0.074205j,3.155816+10.792051j,0.029624+0.023479j,-0.233807-0.127476j,0.321639-0.328123j,0.459473,0.835860,1.147649,0.430120,-0.617913-0.043514j,0.000020-0.317310j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj


In [ ]:
# Calcular coeficientes Z_in y Z_out
df_s_parameters['Z_in'] = 50 * ((1 + df_s_parameters['ro_in']) / (1 - df_s_parameters['ro_in']))
df_s_parameters['Z_out'] = 50 * ((1 + df_s_parameters['ro_out']) / (1 - df_s_parameters['ro_out']))


print("DataFrame con las nuevas columnas de coeficientes Z_in y Z_out:")
display(df_s_parameters)

DataFrame con las nuevas columnas de coeficientes Z_in y Z_out:


,filename,network_name,frequency_ghz,VCE,IC,s11_mag,s21_mag,s12_mag,s22_mag,s11_deg,s21_deg,s12_deg,s22_deg,s11_complex,s21_complex,s12_complex,s22_complex,delta,delta_modulo,K,B1,B2,C1,C2,ro_s(m),ro_L(m),ro_in,ro_out,Z_in,Z_out
0,BFP760_VCE_2.0V_IC_50mA.s2p,BFP760_VCE_2.0V_IC_50mA,2.2,2.0,50.0,0.6744,10.485,0.0384,0.2792,171.7,72.5,37.7,-157.8,-0.667336+0.097354j,3.152900+ 9.999722j,0.030383+0.023483j,-0.258503-0.105493j,0.321804-0.332627j,0.462816,0.846237,1.162664,0.408939,-0.619239-0.022579j,-0.011369-0.296138j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj
1,BFP760_w_noise_VCE_3.0V_IC_4.0mA.s2p,BFP760_w_noise_VCE_3.0V_IC_4.0mA,2.2,3.0,4.0,0.7214,7.122,0.0992,0.5460,-127.9,93.9,18.0,-85.0,-0.443145-0.569245j,-0.484405+ 7.105507j,0.094345+0.030654j,0.047587-0.543922j,-0.067196-0.441571j,0.446654,0.269614,1.022802,0.578198,-0.680128-0.511683j,-0.233553-0.701351j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj
2,BFP760_VCE_4.0V_IC_28mA.s2p,BFP760_VCE_4.0V_IC_28mA,2.2,4.0,28.0,0.6461,11.184,0.0429,0.2851,-179.0,76.3,30.7,-140.8,-0.646002-0.011276j,2.648798+10.865805j,0.036888+0.021902j,-0.220937-0.180192j,0.280972-0.339934j,0.441021,0.725075,1.141663,0.469337,-0.645178-0.137009j,-0.043262-0.402957j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj
3,BFP760_w_noise_VCE_2.0V_IC_10mA.s2p,BFP760_w_noise_VCE_2.0V_IC_10mA,2.2,2.0,10.0,0.6636,9.407,0.0669,0.3836,-158.6,82.6,18.1,-119.0,-0.617849-0.242132j,1.211580+ 9.328651j,0.063590+0.020784j,-0.185973-0.335504j,0.150512-0.366065j,0.395800,0.452183,1.136558,0.550126,-0.712674-0.360708j,-0.181616-0.598121j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj
4,BFP760_VCE_1.5V_IC_60mA.s2p,BFP760_VCE_1.5V_IC_60mA,2.2,1.5,60.0,0.7895,5.356,0.0357,0.1920,163.9,68.5,37.0,-166.0,-0.758535+0.218940j,1.962981+ 4.983316j,0.028511+0.021485j,-0.186297-0.046449j,0.202581-0.189810j,0.277609,1.090147,1.509380,0.336487,-0.729611+0.174169j,0.008925-0.146073j,-0.870981-0.207916j,0.035522+0.581403j,-0.870981+0.207916j,0.035522-0.581403j,2.795898+5.867048j,26.048096-45.843039j
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,BFP760_w_noise_VCE_2.0V_IC_35mA.s2p,BFP760_w_noise_VCE_2.0V_IC_35mA,2.2,2.0,35.0,0.6601,10.813,0.0415,0.2986,176.2,74.2,33.2,-152.9,-0.658649+0.043747j,2.944166+10.404463j,0.034726+0.022724j,-0.265818-0.136026j,0.315223-0.350241j,0.471205,0.776776,1.124536,0.431396,-0.622499-0.092231j,-0.042874-0.352921j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj
150,BFP760_VCE_3.0V_IC_70mA.s2p,BFP760_VCE_3.0V_IC_70mA,2.2,3.0,70.0,0.6913,9.889,0.0351,0.2313,168.5,71.6,41.9,-153.8,-0.677422+0.137823j,3.121453+ 9.383435j,0.026125+0.023441j,-0.207536-0.102120j,0.293071-0.277740j,0.403770,0.909864,1.261366,0.412574,-0.644962+0.050254j,0.029276-0.249875j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj
151,BFP760_VCE_3.5V_IC_22mA.s2p,BFP760_VCE_3.5V_IC_22mA,2.2,3.5,22.0,0.6482,10.919,0.0466,0.3025,-175.2,77.4,27.4,-136.5,-0.645927-0.054240j,2.381906+10.656035j,0.041372+0.021445j,-0.219426-0.208227j,0.260416-0.345543j,0.432685,0.663830,1.141440,0.484126,-0.660736-0.184287j,-0.069958-0.445548j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj
152,BFP760_VCE_3.5V_IC_50mA.s2p,BFP760_VCE_3.5V_IC_50mA,2.2,3.5,50.0,0.6555,11.244,0.0378,0.2663,173.5,73.7,38.4,-151.4,-0.651286+0.074205j,3.155816+10.792051j,0.029624+0.023479j,-0.233807-0.127476j,0.321639-0.328123j,0.459473,0.835860,1.147649,0.430120,-0.617913-0.043514j,0.000020-0.317310j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj


In [ ]:
filtered_df = df_s_parameters[(df_s_parameters['delta'] < 1) & (df_s_parameters['K'] > 1)]

print("Filas donde delta < 1 y K > 1:")
display(filtered_df)

Filas donde delta < 1 y K > 1:


,filename,network_name,frequency_ghz,VCE,IC,s11_mag,s21_mag,s12_mag,s22_mag,s11_deg,s21_deg,s12_deg,s22_deg,s11_complex,s21_complex,s12_complex,s22_complex,delta,delta_modulo,K,B1,B2,C1,C2,ro_s(m),ro_L(m),ro_in,ro_out,Z_in,Z_out
4,BFP760_VCE_1.5V_IC_60mA.s2p,BFP760_VCE_1.5V_IC_60mA,2.2,1.5,60.0,0.7895,5.356,0.0357,0.1920,163.9,68.5,37.0,-166.0,-0.758535+0.218940j,1.962981+4.983316j,0.028511+0.021485j,-0.186297-0.046449j,0.202581-0.189810j,0.277609,1.090147,1.509380,0.336487,-0.729611+0.174169j,0.008925-0.146073j,-0.870981-0.207916j,0.035522+0.581403j,-0.870981+0.207916j,0.035522-0.581403j,2.795898+5.867048j,26.048096-45.843039j
7,BFP760_VCE_1.0V_IC_55mA.s2p,BFP760_VCE_1.0V_IC_55mA,2.2,1.0,55.0,0.8468,2.617,0.0388,0.2446,162.1,67.2,33.5,172.7,-0.805810+0.260270j,1.014128+2.412516j,0.032355+0.021415j,-0.242617+0.031080j,0.206267-0.187965j,0.279064,1.482069,1.579364,0.264882,-0.749924+0.221077j,-0.027484-0.066699j,-0.832553-0.245436j,-0.112865+0.273905j,-0.832553+0.245436j,-0.112865-0.273905j,3.607091+7.179655j,34.725642-20.853141j
21,BFP760_VCE_2.0V_IC_70mA.s2p,BFP760_VCE_2.0V_IC_70mA,2.2,2.0,70.0,0.7915,5.262,0.0337,0.1558,163.7,68.9,39.5,-155.5,-0.759686+0.222148j,1.894303+4.909202j,0.026004+0.021436j,-0.141772-0.064609j,0.178029-0.150675j,0.233232,1.138140,1.547801,0.343404,-0.744181+0.189284j,0.026946-0.139526j,-0.855111-0.217499j,0.100517+0.520474j,-0.855111+0.217499j,0.100517-0.520474j,3.174194+6.234310j,33.288359-48.193671j
24,BFP760_VCE_1.0V_IC_50mA.s2p,BFP760_VCE_1.0V_IC_50mA,2.2,1.0,50.0,0.8110,4.290,0.0394,0.2570,163.2,67.7,32.5,179.1,-0.776386+0.234405j,1.627867+3.969150j,0.033230+0.021170j,-0.256968+0.004037j,0.228492-0.229723j,0.324009,1.127671,1.486690,0.303346,-0.716743+0.176296j,-0.025722-0.120758j,-0.862050-0.212036j,-0.107278+0.503642j,-0.862050+0.212036j,-0.107278-0.503642j,3.016777+6.037152j,24.830247-34.036340j
30,BFP760_VCE_1.0V_IC_60mA.s2p,BFP760_VCE_1.0V_IC_60mA,2.2,1.0,60.0,0.8655,1.886,0.0379,0.2516,160.7,65.9,35.2,169.0,-0.816860+0.286060j,0.770111+1.721605j,0.030970+0.021847j,-0.246977+0.048008j,0.201774-0.180008j,0.270399,1.823763,1.612672,0.241097,-0.758384+0.251289j,-0.030663-0.041314j,-0.828515-0.274527j,-0.133570+0.179967j,-0.828515+0.274527j,-0.133570-0.179967j,3.483608+8.029837j,36.048041-13.661082j
62,BFP760_VCE_1.0V_IC_65mA.s2p,BFP760_VCE_1.0V_IC_65mA,2.2,1.0,65.0,0.8756,1.627,0.0373,0.2604,159.3,64.3,37.0,167.9,-0.819075+0.309503j,0.705563+1.466052j,0.029789+0.022448j,-0.254615+0.054585j,0.203546-0.183024j,0.273731,1.981022,1.623939,0.226204,-0.757259+0.274013j,-0.031249-0.032327j,-0.826927-0.299222j,-0.144083+0.149054j,-0.826927+0.299222j,-0.144083-0.149054j,3.306758+8.730808j,35.947398-11.197437j
107,BFP760_VCE_1.0V_IC_70mA.s2p,BFP760_VCE_1.0V_IC_70mA,2.2,1.0,70.0,0.8826,1.507,0.0369,0.2691,158.1,62.9,38.0,167.7,-0.818908+0.329199j,0.686506+1.341551j,0.029078+0.022718j,-0.262923+0.057326j,0.206953-0.188104j,0.279666,2.039401,1.628355,0.215219,-0.753712+0.291606j,-0.031524-0.028585j,-0.825519-0.319388j,-0.152695+0.138460j,-0.825519+0.319388j,-0.152695-0.138460j,3.151953+9.299315j,35.519317-10.272453j
109,BFP760_VCE_1.5V_IC_70mA.s2p,BFP760_VCE_1.5V_IC_70mA,2.2,1.5,70.0,0.8478,2.677,0.0346,0.1600,161.4,67.4,38.2,-175.1,-0.803518+0.270414j,1.028759+2.471434j,0.027191+0.021397j,-0.159415-0.013667j,0.156697-0.121339j,0.198185,1.591983,1.653888,0.267558,-0.780196+0.248929j,-0.000695-0.068792j,-0.828517-0.264346j,-0.002795+0.276812j,-0.828517+0.264346j,-0.002795-0.276812j,3.569532+7.744467j,42.660709-25.578088j
112,BFP760_VCE_1.5V_IC_65mA.s2p,BFP760_VCE_1.5V_IC_65mA,2.2,1.5,65.0,0.8257,3.637,0.0351,0.1650,162.6,68.1,37.2,-169.8,-0.787916+0.246918j,1.356557+3.374540j,0.027958+0.021221j,-0.162392-0.029219j,0.168852-0.140210j,0.219476,1.328402,1.606386,0.297275,-0.764593+0.219215j,0.005269-0.098000j,-0.835782-0.239626j,0.020245+0.376535j,-0.835782+0.239626j,0.020245-0.376535j,3.560129+6.991241j,38.931350-34.177663j


In [ ]:
# Calcular coeficientes R_in y R_out, X_in y X_out
df_s_parameters['R_in'] = df_s_parameters['Z_in'].apply(lambda x: x.real)
df_s_parameters['X_in'] = df_s_parameters['Z_in'].apply(lambda x: x.imag)
df_s_parameters['R_out'] = df_s_parameters['Z_out'].apply(lambda x: x.real)
df_s_parameters['X_out'] = df_s_parameters['Z_out'].apply(lambda x: x.imag)

print("DataFrame con las nuevas columnas de coeficientes R_in y R_out, X_in y X_out:")
display(df_s_parameters)

DataFrame con las nuevas columnas de coeficientes R_in y R_out, X_in y X_out:


,filename,network_name,frequency_ghz,VCE,IC,s11_mag,s21_mag,s12_mag,s22_mag,s11_deg,s21_deg,s12_deg,s22_deg,s11_complex,s21_complex,s12_complex,s22_complex,delta,delta_modulo,K,B1,B2,C1,C2,ro_s(m),ro_L(m),ro_in,ro_out,Z_in,Z_out,R_in,X_in,R_out,X_out
0,BFP760_VCE_2.0V_IC_50mA.s2p,BFP760_VCE_2.0V_IC_50mA,2.2,2.0,50.0,0.6744,10.485,0.0384,0.2792,171.7,72.5,37.7,-157.8,-0.667336+0.097354j,3.152900+ 9.999722j,0.030383+0.023483j,-0.258503-0.105493j,0.321804-0.332627j,0.462816,0.846237,1.162664,0.408939,-0.619239-0.022579j,-0.011369-0.296138j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN
1,BFP760_w_noise_VCE_3.0V_IC_4.0mA.s2p,BFP760_w_noise_VCE_3.0V_IC_4.0mA,2.2,3.0,4.0,0.7214,7.122,0.0992,0.5460,-127.9,93.9,18.0,-85.0,-0.443145-0.569245j,-0.484405+ 7.105507j,0.094345+0.030654j,0.047587-0.543922j,-0.067196-0.441571j,0.446654,0.269614,1.022802,0.578198,-0.680128-0.511683j,-0.233553-0.701351j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN
2,BFP760_VCE_4.0V_IC_28mA.s2p,BFP760_VCE_4.0V_IC_28mA,2.2,4.0,28.0,0.6461,11.184,0.0429,0.2851,-179.0,76.3,30.7,-140.8,-0.646002-0.011276j,2.648798+10.865805j,0.036888+0.021902j,-0.220937-0.180192j,0.280972-0.339934j,0.441021,0.725075,1.141663,0.469337,-0.645178-0.137009j,-0.043262-0.402957j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN
3,BFP760_w_noise_VCE_2.0V_IC_10mA.s2p,BFP760_w_noise_VCE_2.0V_IC_10mA,2.2,2.0,10.0,0.6636,9.407,0.0669,0.3836,-158.6,82.6,18.1,-119.0,-0.617849-0.242132j,1.211580+ 9.328651j,0.063590+0.020784j,-0.185973-0.335504j,0.150512-0.366065j,0.395800,0.452183,1.136558,0.550126,-0.712674-0.360708j,-0.181616-0.598121j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN
4,BFP760_VCE_1.5V_IC_60mA.s2p,BFP760_VCE_1.5V_IC_60mA,2.2,1.5,60.0,0.7895,5.356,0.0357,0.1920,163.9,68.5,37.0,-166.0,-0.758535+0.218940j,1.962981+ 4.983316j,0.028511+0.021485j,-0.186297-0.046449j,0.202581-0.189810j,0.277609,1.090147,1.509380,0.336487,-0.729611+0.174169j,0.008925-0.146073j,-0.870981-0.207916j,0.035522+0.581403j,-0.870981+0.207916j,0.035522-0.581403j,2.795898+5.867048j,26.048096-45.843039j,2.795898,5.867048,26.048096,-45.843039
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,BFP760_w_noise_VCE_2.0V_IC_35mA.s2p,BFP760_w_noise_VCE_2.0V_IC_35mA,2.2,2.0,35.0,0.6601,10.813,0.0415,0.2986,176.2,74.2,33.2,-152.9,-0.658649+0.043747j,2.944166+10.404463j,0.034726+0.022724j,-0.265818-0.136026j,0.315223-0.350241j,0.471205,0.776776,1.124536,0.431396,-0.622499-0.092231j,-0.042874-0.352921j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN
150,BFP760_VCE_3.0V_IC_70mA.s2p,BFP760_VCE_3.0V_IC_70mA,2.2,3.0,70.0,0.6913,9.889,0.0351,0.2313,168.5,71.6,41.9,-153.8,-0.677422+0.137823j,3.121453+ 9.383435j,0.026125+0.023441j,-0.207536-0.102120j,0.293071-0.277740j,0.403770,0.909864,1.261366,0.412574,-0.644962+0.050254j,0.029276-0.249875j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN
151,BFP760_VCE_3.5V_IC_22mA.s2p,BFP760_VCE_3.5V_IC_22mA,2.2,3.5,22.0,0.6482,10.919,0.0466,0.3025,-175.2,77.4,27.4,-136.5,-0.645927-0.054240j,2.381906+10.656035j,0.041372+0.021445j,-0.219426-0.208227j,0.260416-0.345543j,0.432685,0.663830,1.141440,0.484126,-0.660736-0.184287j,-0.069958-0.445548j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN
152,BFP760_VCE_3.5V_IC_50mA.s2p,BFP760_VCE_3.5V_IC_50mA,2.2,3.5,50.0,0.6555,11.244,0.0378,0.2663,173.5,73.7,38.4,-151.4,-0.651286+0.074205j,3.155816+10.792051j,0.029624+0.023479j,-0.233807-0.127476j,0.321639-0.328123j,0.459473,0.835860,1.147649,0.430120,-0.617913-0.043514j,0.000020-0.317310j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN


In [ ]:
# Calcular coeficientes R_in_p y X_in_p
df_s_parameters['R_in_p'] = (df_s_parameters['R_in'] * (1 + ((df_s_parameters['X_in'] / df_s_parameters['R_in'])**2) ))
df_s_parameters['X_in_p'] = (df_s_parameters['X_in'] * (1 + ((df_s_parameters['R_in'] / df_s_parameters['X_in'])**2) ))


print("DataFrame con las nuevas columnas de coeficientes R_in_p y X_in_p:")
display(df_s_parameters)

DataFrame con las nuevas columnas de coeficientes R_in_p y X_in_p:


,filename,network_name,frequency_ghz,VCE,IC,s11_mag,s21_mag,s12_mag,s22_mag,s11_deg,s21_deg,s12_deg,s22_deg,s11_complex,s21_complex,s12_complex,s22_complex,delta,delta_modulo,K,B1,B2,C1,C2,ro_s(m),ro_L(m),ro_in,ro_out,Z_in,Z_out,R_in,X_in,R_out,X_out,R_in_p,X_in_p
0,BFP760_VCE_2.0V_IC_50mA.s2p,BFP760_VCE_2.0V_IC_50mA,2.2,2.0,50.0,0.6744,10.485,0.0384,0.2792,171.7,72.5,37.7,-157.8,-0.667336+0.097354j,3.152900+ 9.999722j,0.030383+0.023483j,-0.258503-0.105493j,0.321804-0.332627j,0.462816,0.846237,1.162664,0.408939,-0.619239-0.022579j,-0.011369-0.296138j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN
1,BFP760_w_noise_VCE_3.0V_IC_4.0mA.s2p,BFP760_w_noise_VCE_3.0V_IC_4.0mA,2.2,3.0,4.0,0.7214,7.122,0.0992,0.5460,-127.9,93.9,18.0,-85.0,-0.443145-0.569245j,-0.484405+ 7.105507j,0.094345+0.030654j,0.047587-0.543922j,-0.067196-0.441571j,0.446654,0.269614,1.022802,0.578198,-0.680128-0.511683j,-0.233553-0.701351j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN
2,BFP760_VCE_4.0V_IC_28mA.s2p,BFP760_VCE_4.0V_IC_28mA,2.2,4.0,28.0,0.6461,11.184,0.0429,0.2851,-179.0,76.3,30.7,-140.8,-0.646002-0.011276j,2.648798+10.865805j,0.036888+0.021902j,-0.220937-0.180192j,0.280972-0.339934j,0.441021,0.725075,1.141663,0.469337,-0.645178-0.137009j,-0.043262-0.402957j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN
3,BFP760_w_noise_VCE_2.0V_IC_10mA.s2p,BFP760_w_noise_VCE_2.0V_IC_10mA,2.2,2.0,10.0,0.6636,9.407,0.0669,0.3836,-158.6,82.6,18.1,-119.0,-0.617849-0.242132j,1.211580+ 9.328651j,0.063590+0.020784j,-0.185973-0.335504j,0.150512-0.366065j,0.395800,0.452183,1.136558,0.550126,-0.712674-0.360708j,-0.181616-0.598121j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN
4,BFP760_VCE_1.5V_IC_60mA.s2p,BFP760_VCE_1.5V_IC_60mA,2.2,1.5,60.0,0.7895,5.356,0.0357,0.1920,163.9,68.5,37.0,-166.0,-0.758535+0.218940j,1.962981+ 4.983316j,0.028511+0.021485j,-0.186297-0.046449j,0.202581-0.189810j,0.277609,1.090147,1.509380,0.336487,-0.729611+0.174169j,0.008925-0.146073j,-0.870981-0.207916j,0.035522+0.581403j,-0.870981+0.207916j,0.035522-0.581403j,2.795898+5.867048j,26.048096-45.843039j,2.795898,5.867048,26.048096,-45.843039,15.107596,7.199412
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,BFP760_w_noise_VCE_2.0V_IC_35mA.s2p,BFP760_w_noise_VCE_2.0V_IC_35mA,2.2,2.0,35.0,0.6601,10.813,0.0415,0.2986,176.2,74.2,33.2,-152.9,-0.658649+0.043747j,2.944166+10.404463j,0.034726+0.022724j,-0.265818-0.136026j,0.315223-0.350241j,0.471205,0.776776,1.124536,0.431396,-0.622499-0.092231j,-0.042874-0.352921j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN
150,BFP760_VCE_3.0V_IC_70mA.s2p,BFP760_VCE_3.0V_IC_70mA,2.2,3.0,70.0,0.6913,9.889,0.0351,0.2313,168.5,71.6,41.9,-153.8,-0.677422+0.137823j,3.121453+ 9.383435j,0.026125+0.023441j,-0.207536-0.102120j,0.293071-0.277740j,0.403770,0.909864,1.261366,0.412574,-0.644962+0.050254j,0.029276-0.249875j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN
151,BFP760_VCE_3.5V_IC_22mA.s2p,BFP760_VCE_3.5V_IC_22mA,2.2,3.5,22.0,0.6482,10.919,0.0466,0.3025,-175.2,77.4,27.4,-136.5,-0.645927-0.054240j,2.381906+10.656035j,0.041372+0.021445j,-0.219426-0.208227j,0.260416-0.345543j,0.432685,0.663830,1.141440,0.484126,-0.660736-0.184287j,-0.069958-0.445548j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN
152,BFP760_VCE_3.5V_IC_50mA.s2p,BFP760_VCE_3.5V_IC_50mA,2.2,3.5,50.0,0.6555,11.244,0.0378,0.2663,173.5,73.7,38.4,-151.4,-0.651286+0.074205j,3.155816+10.792051j,0.029624+0.023479j,-0.233807-0.127476j,0.321639-0.328123j,0.459473,0.835860,1.147649,0.430120,-0.617913-0.043514j,0.000020-0.317310j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Calcular C_in
df_s_parameters['C_in'] = (1 / (2 * np.pi * target_frequency * (df_s_parameters['X_in_p'])))
df_s_parameters['Z_0_QWin'] = np.sqrt(50 * df_s_parameters['R_in_p'])
df_s_parameters['Z_0_QWout'] = np.sqrt(50 * df_s_parameters['R_out'])


print("DataFrame con las nuevas columnas de C_in y Z_0_QWin:")
display(df_s_parameters)

DataFrame con las nuevas columnas de C_in y Z_0_QWin:


,filename,network_name,frequency_ghz,VCE,IC,s11_mag,s21_mag,s12_mag,s22_mag,s11_deg,s21_deg,s12_deg,s22_deg,s11_complex,s21_complex,s12_complex,s22_complex,delta,delta_modulo,K,B1,B2,C1,C2,ro_s(m),ro_L(m),ro_in,ro_out,Z_in,Z_out,R_in,X_in,R_out,X_out,R_in_p,X_in_p,C_in,Z_0_QWin,Z_0_QWout
0,BFP760_VCE_2.0V_IC_50mA.s2p,BFP760_VCE_2.0V_IC_50mA,2.2,2.0,50.0,0.6744,10.485,0.0384,0.2792,171.7,72.5,37.7,-157.8,-0.667336+0.097354j,3.152900+ 9.999722j,0.030383+0.023483j,-0.258503-0.105493j,0.321804-0.332627j,0.462816,0.846237,1.162664,0.408939,-0.619239-0.022579j,-0.011369-0.296138j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,BFP760_w_noise_VCE_3.0V_IC_4.0mA.s2p,BFP760_w_noise_VCE_3.0V_IC_4.0mA,2.2,3.0,4.0,0.7214,7.122,0.0992,0.5460,-127.9,93.9,18.0,-85.0,-0.443145-0.569245j,-0.484405+ 7.105507j,0.094345+0.030654j,0.047587-0.543922j,-0.067196-0.441571j,0.446654,0.269614,1.022802,0.578198,-0.680128-0.511683j,-0.233553-0.701351j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BFP760_VCE_4.0V_IC_28mA.s2p,BFP760_VCE_4.0V_IC_28mA,2.2,4.0,28.0,0.6461,11.184,0.0429,0.2851,-179.0,76.3,30.7,-140.8,-0.646002-0.011276j,2.648798+10.865805j,0.036888+0.021902j,-0.220937-0.180192j,0.280972-0.339934j,0.441021,0.725075,1.141663,0.469337,-0.645178-0.137009j,-0.043262-0.402957j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BFP760_w_noise_VCE_2.0V_IC_10mA.s2p,BFP760_w_noise_VCE_2.0V_IC_10mA,2.2,2.0,10.0,0.6636,9.407,0.0669,0.3836,-158.6,82.6,18.1,-119.0,-0.617849-0.242132j,1.211580+ 9.328651j,0.063590+0.020784j,-0.185973-0.335504j,0.150512-0.366065j,0.395800,0.452183,1.136558,0.550126,-0.712674-0.360708j,-0.181616-0.598121j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,BFP760_VCE_1.5V_IC_60mA.s2p,BFP760_VCE_1.5V_IC_60mA,2.2,1.5,60.0,0.7895,5.356,0.0357,0.1920,163.9,68.5,37.0,-166.0,-0.758535+0.218940j,1.962981+ 4.983316j,0.028511+0.021485j,-0.186297-0.046449j,0.202581-0.189810j,0.277609,1.090147,1.509380,0.336487,-0.729611+0.174169j,0.008925-0.146073j,-0.870981-0.207916j,0.035522+0.581403j,-0.870981+0.207916j,0.035522-0.581403j,2.795898+5.867048j,26.048096-45.843039j,2.795898,5.867048,26.048096,-45.843039,15.107596,7.199412,1.004848e-11,27.484173,36.088846
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,BFP760_w_noise_VCE_2.0V_IC_35mA.s2p,BFP760_w_noise_VCE_2.0V_IC_35mA,2.2,2.0,35.0,0.6601,10.813,0.0415,0.2986,176.2,74.2,33.2,-152.9,-0.658649+0.043747j,2.944166+10.404463j,0.034726+0.022724j,-0.265818-0.136026j,0.315223-0.350241j,0.471205,0.776776,1.124536,0.431396,-0.622499-0.092231j,-0.042874-0.352921j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
150,BFP760_VCE_3.0V_IC_70mA.s2p,BFP760_VCE_3.0V_IC_70mA,2.2,3.0,70.0,0.6913,9.889,0.0351,0.2313,168.5,71.6,41.9,-153.8,-0.677422+0.137823j,3.121453+ 9.383435j,0.026125+0.023441j,-0.207536-0.102120j,0.293071-0.277740j,0.403770,0.909864,1.261366,0.412574,-0.644962+0.050254j,0.029276-0.249875j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
151,BFP760_VCE_3.5V_IC_22mA.s2p,BFP760_VCE_3.5V_IC_22mA,2.2,3.5,22.0,0.6482,10.919,0.0466,0.3025,-175.2,77.4,27.4,-136.5,-0.645927-0.054240j,2.381906+10.656035j,0.041372+0.021445j,-0.219426-0.208227j,0.260416-0.345543j,0.432685,0.663830,1.141440,0.484126,-0.660736-0.184287j,-0.069958-0.445548j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
152,BFP760_VCE_3.5V_IC_50mA.s2p,BFP760_VCE_3.5V_IC_50mA,2.2,3.5,50.0,0.6555,11.244,0.0378,0.2663,173.5,73.7,38.4,-151.4,-0.651286+0.074205j,3.155816+10.792051j,0.029624+0.023479j,-0.233807-0.127476j,0.321639-0.328123j,0.459473,0.835860,1.147649,0.430120,-0.617913-0.043514j,0.00002

In [ ]:
# Calcular X_L_p
df_s_parameters['X_L_p'] = ((df_s_parameters['Z_0_QWout'])**2)/(-df_s_parameters['X_out'])


print("DataFrame con las nuevas columnas de X_L_p:")
display(df_s_parameters)

DataFrame con las nuevas columnas de X_L_p:


,filename,network_name,frequency_ghz,VCE,IC,s11_mag,s21_mag,s12_mag,s22_mag,s11_deg,s21_deg,s12_deg,s22_deg,s11_complex,s21_complex,s12_complex,s22_complex,delta,delta_modulo,K,B1,B2,C1,C2,ro_s(m),ro_L(m),ro_in,ro_out,Z_in,Z_out,R_in,X_in,R_out,X_out,R_in_p,X_in_p,C_in,Z_0_QWin,Z_0_QWout,X_L_p
0,BFP760_VCE_2.0V_IC_50mA.s2p,BFP760_VCE_2.0V_IC_50mA,2.2,2.0,50.0,0.6744,10.485,0.0384,0.2792,171.7,72.5,37.7,-157.8,-0.667336+0.097354j,3.152900+ 9.999722j,0.030383+0.023483j,-0.258503-0.105493j,0.321804-0.332627j,0.462816,0.846237,1.162664,0.408939,-0.619239-0.022579j,-0.011369-0.296138j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,BFP760_w_noise_VCE_3.0V_IC_4.0mA.s2p,BFP760_w_noise_VCE_3.0V_IC_4.0mA,2.2,3.0,4.0,0.7214,7.122,0.0992,0.5460,-127.9,93.9,18.0,-85.0,-0.443145-0.569245j,-0.484405+ 7.105507j,0.094345+0.030654j,0.047587-0.543922j,-0.067196-0.441571j,0.446654,0.269614,1.022802,0.578198,-0.680128-0.511683j,-0.233553-0.701351j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BFP760_VCE_4.0V_IC_28mA.s2p,BFP760_VCE_4.0V_IC_28mA,2.2,4.0,28.0,0.6461,11.184,0.0429,0.2851,-179.0,76.3,30.7,-140.8,-0.646002-0.011276j,2.648798+10.865805j,0.036888+0.021902j,-0.220937-0.180192j,0.280972-0.339934j,0.441021,0.725075,1.141663,0.469337,-0.645178-0.137009j,-0.043262-0.402957j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BFP760_w_noise_VCE_2.0V_IC_10mA.s2p,BFP760_w_noise_VCE_2.0V_IC_10mA,2.2,2.0,10.0,0.6636,9.407,0.0669,0.3836,-158.6,82.6,18.1,-119.0,-0.617849-0.242132j,1.211580+ 9.328651j,0.063590+0.020784j,-0.185973-0.335504j,0.150512-0.366065j,0.395800,0.452183,1.136558,0.550126,-0.712674-0.360708j,-0.181616-0.598121j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,BFP760_VCE_1.5V_IC_60mA.s2p,BFP760_VCE_1.5V_IC_60mA,2.2,1.5,60.0,0.7895,5.356,0.0357,0.1920,163.9,68.5,37.0,-166.0,-0.758535+0.218940j,1.962981+ 4.983316j,0.028511+0.021485j,-0.186297-0.046449j,0.202581-0.189810j,0.277609,1.090147,1.509380,0.336487,-0.729611+0.174169j,0.008925-0.146073j,-0.870981-0.207916j,0.035522+0.581403j,-0.870981+0.207916j,0.035522-0.581403j,2.795898+5.867048j,26.048096-45.843039j,2.795898,5.867048,26.048096,-45.843039,15.107596,7.199412,1.004848e-11,27.484173,36.088846,28.410089
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,BFP760_w_noise_VCE_2.0V_IC_35mA.s2p,BFP760_w_noise_VCE_2.0V_IC_35mA,2.2,2.0,35.0,0.6601,10.813,0.0415,0.2986,176.2,74.2,33.2,-152.9,-0.658649+0.043747j,2.944166+10.404463j,0.034726+0.022724j,-0.265818-0.136026j,0.315223-0.350241j,0.471205,0.776776,1.124536,0.431396,-0.622499-0.092231j,-0.042874-0.352921j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
150,BFP760_VCE_3.0V_IC_70mA.s2p,BFP760_VCE_3.0V_IC_70mA,2.2,3.0,70.0,0.6913,9.889,0.0351,0.2313,168.5,71.6,41.9,-153.8,-0.677422+0.137823j,3.121453+ 9.383435j,0.026125+0.023441j,-0.207536-0.102120j,0.293071-0.277740j,0.403770,0.909864,1.261366,0.412574,-0.644962+0.050254j,0.029276-0.249875j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
151,BFP760_VCE_3.5V_IC_22mA.s2p,BFP760_VCE_3.5V_IC_22mA,2.2,3.5,22.0,0.6482,10.919,0.0466,0.3025,-175.2,77.4,27.4,-136.5,-0.645927-0.054240j,2.381906+10.656035j,0.041372+0.021445j,-0.219426-0.208227j,0.260416-0.345543j,0.432685,0.663830,1.141440,0.484126,-0.660736-0.184287j,-0.069958-0.445548j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
152,BFP760_VCE_3.5V_IC_50mA.s2p,BFP760_VCE_3.5V_IC_50mA,2.2,3.5,50.0,0.6555,11.244,0.0378,0.2663,173.5,73.7,38.4,-151.4,-0.651286+0.074205j,3.155816+10.792051j,0.029624+0.023479j,-0.233807-0.127476j,0.321639-0.328123j,0.459473,0.8358

In [ ]:
# Calcular C_out
df_s_parameters['C_out'] = (1 / (2 * np.pi * target_frequency * (df_s_parameters['X_L_p'])))


print("DataFrame con las nuevas columnas de C_out:")
display(df_s_parameters)

DataFrame con las nuevas columnas de C_out:


,filename,network_name,frequency_ghz,VCE,IC,s11_mag,s21_mag,s12_mag,s22_mag,s11_deg,s21_deg,s12_deg,s22_deg,s11_complex,s21_complex,s12_complex,s22_complex,delta,delta_modulo,K,B1,B2,C1,C2,ro_s(m),ro_L(m),ro_in,ro_out,Z_in,Z_out,R_in,X_in,R_out,X_out,R_in_p,X_in_p,C_in,Z_0_QWin,Z_0_QWout,X_L_p,C_out
0,BFP760_VCE_2.0V_IC_50mA.s2p,BFP760_VCE_2.0V_IC_50mA,2.2,2.0,50.0,0.6744,10.485,0.0384,0.2792,171.7,72.5,37.7,-157.8,-0.667336+0.097354j,3.152900+ 9.999722j,0.030383+0.023483j,-0.258503-0.105493j,0.321804-0.332627j,0.462816,0.846237,1.162664,0.408939,-0.619239-0.022579j,-0.011369-0.296138j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,BFP760_w_noise_VCE_3.0V_IC_4.0mA.s2p,BFP760_w_noise_VCE_3.0V_IC_4.0mA,2.2,3.0,4.0,0.7214,7.122,0.0992,0.5460,-127.9,93.9,18.0,-85.0,-0.443145-0.569245j,-0.484405+ 7.105507j,0.094345+0.030654j,0.047587-0.543922j,-0.067196-0.441571j,0.446654,0.269614,1.022802,0.578198,-0.680128-0.511683j,-0.233553-0.701351j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BFP760_VCE_4.0V_IC_28mA.s2p,BFP760_VCE_4.0V_IC_28mA,2.2,4.0,28.0,0.6461,11.184,0.0429,0.2851,-179.0,76.3,30.7,-140.8,-0.646002-0.011276j,2.648798+10.865805j,0.036888+0.021902j,-0.220937-0.180192j,0.280972-0.339934j,0.441021,0.725075,1.141663,0.469337,-0.645178-0.137009j,-0.043262-0.402957j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BFP760_w_noise_VCE_2.0V_IC_10mA.s2p,BFP760_w_noise_VCE_2.0V_IC_10mA,2.2,2.0,10.0,0.6636,9.407,0.0669,0.3836,-158.6,82.6,18.1,-119.0,-0.617849-0.242132j,1.211580+ 9.328651j,0.063590+0.020784j,-0.185973-0.335504j,0.150512-0.366065j,0.395800,0.452183,1.136558,0.550126,-0.712674-0.360708j,-0.181616-0.598121j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,BFP760_VCE_1.5V_IC_60mA.s2p,BFP760_VCE_1.5V_IC_60mA,2.2,1.5,60.0,0.7895,5.356,0.0357,0.1920,163.9,68.5,37.0,-166.0,-0.758535+0.218940j,1.962981+ 4.983316j,0.028511+0.021485j,-0.186297-0.046449j,0.202581-0.189810j,0.277609,1.090147,1.509380,0.336487,-0.729611+0.174169j,0.008925-0.146073j,-0.870981-0.207916j,0.035522+0.581403j,-0.870981+0.207916j,0.035522-0.581403j,2.795898+5.867048j,26.048096-45.843039j,2.795898,5.867048,26.048096,-45.843039,15.107596,7.199412,1.004848e-11,27.484173,36.088846,28.410089,2.546390e-12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,BFP760_w_noise_VCE_2.0V_IC_35mA.s2p,BFP760_w_noise_VCE_2.0V_IC_35mA,2.2,2.0,35.0,0.6601,10.813,0.0415,0.2986,176.2,74.2,33.2,-152.9,-0.658649+0.043747j,2.944166+10.404463j,0.034726+0.022724j,-0.265818-0.136026j,0.315223-0.350241j,0.471205,0.776776,1.124536,0.431396,-0.622499-0.092231j,-0.042874-0.352921j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
150,BFP760_VCE_3.0V_IC_70mA.s2p,BFP760_VCE_3.0V_IC_70mA,2.2,3.0,70.0,0.6913,9.889,0.0351,0.2313,168.5,71.6,41.9,-153.8,-0.677422+0.137823j,3.121453+ 9.383435j,0.026125+0.023441j,-0.207536-0.102120j,0.293071-0.277740j,0.403770,0.909864,1.261366,0.412574,-0.644962+0.050254j,0.029276-0.249875j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
151,BFP760_VCE_3.5V_IC_22mA.s2p,BFP760_VCE_3.5V_IC_22mA,2.2,3.5,22.0,0.6482,10.919,0.0466,0.3025,-175.2,77.4,27.4,-136.5,-0.645927-0.054240j,2.381906+10.656035j,0.041372+0.021445j,-0.219426-0.208227j,0.260416-0.345543j,0.432685,0.663830,1.141440,0.484126,-0.660736-0.184287j,-0.069958-0.445548j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
152,BFP760_VCE_3.5V_IC_50mA.s2p,BFP760_VCE_3.5V_IC_50mA,2.2,3.5,50.0,0.6555,11.244,0.0378,0.2663,173.5,73.7,38.4,-151.4,-0.651286+0.074205j,3.155816+10.792051j,0.029624+0.023479j,-0.

In [ ]:
# Calcular Gt_max_db
df_s_parameters['Gt_max_db'] = 10 * (np.log10(((np.abs(df_s_parameters['s21_complex']))/(np.abs(df_s_parameters['s12_complex'])))*(df_s_parameters['K'] - (np.sqrt(((df_s_parameters['K'])**2) - 1)))))


print("DataFrame con las nuevas columnas de Gt_max_db:")
display(df_s_parameters)

DataFrame con las nuevas columnas de Gt_max_db:


/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


,filename,network_name,frequency_ghz,VCE,IC,s11_mag,s21_mag,s12_mag,s22_mag,s11_deg,s21_deg,s12_deg,s22_deg,s11_complex,s21_complex,s12_complex,s22_complex,delta,delta_modulo,K,B1,B2,C1,C2,ro_s(m),ro_L(m),ro_in,ro_out,Z_in,Z_out,R_in,X_in,R_out,X_out,R_in_p,X_in_p,C_in,Z_0_QWin,Z_0_QWout,X_L_p,C_out,Gt_max_db
0,BFP760_VCE_2.0V_IC_50mA.s2p,BFP760_VCE_2.0V_IC_50mA,2.2,2.0,50.0,0.6744,10.485,0.0384,0.2792,171.7,72.5,37.7,-157.8,-0.667336+0.097354j,3.152900+ 9.999722j,0.030383+0.023483j,-0.258503-0.105493j,0.321804-0.332627j,0.462816,0.846237,1.162664,0.408939,-0.619239-0.022579j,-0.011369-0.296138j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,BFP760_w_noise_VCE_3.0V_IC_4.0mA.s2p,BFP760_w_noise_VCE_3.0V_IC_4.0mA,2.2,3.0,4.0,0.7214,7.122,0.0992,0.5460,-127.9,93.9,18.0,-85.0,-0.443145-0.569245j,-0.484405+ 7.105507j,0.094345+0.030654j,0.047587-0.543922j,-0.067196-0.441571j,0.446654,0.269614,1.022802,0.578198,-0.680128-0.511683j,-0.233553-0.701351j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BFP760_VCE_4.0V_IC_28mA.s2p,BFP760_VCE_4.0V_IC_28mA,2.2,4.0,28.0,0.6461,11.184,0.0429,0.2851,-179.0,76.3,30.7,-140.8,-0.646002-0.011276j,2.648798+10.865805j,0.036888+0.021902j,-0.220937-0.180192j,0.280972-0.339934j,0.441021,0.725075,1.141663,0.469337,-0.645178-0.137009j,-0.043262-0.402957j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BFP760_w_noise_VCE_2.0V_IC_10mA.s2p,BFP760_w_noise_VCE_2.0V_IC_10mA,2.2,2.0,10.0,0.6636,9.407,0.0669,0.3836,-158.6,82.6,18.1,-119.0,-0.617849-0.242132j,1.211580+ 9.328651j,0.063590+0.020784j,-0.185973-0.335504j,0.150512-0.366065j,0.395800,0.452183,1.136558,0.550126,-0.712674-0.360708j,-0.181616-0.598121j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,BFP760_VCE_1.5V_IC_60mA.s2p,BFP760_VCE_1.5V_IC_60mA,2.2,1.5,60.0,0.7895,5.356,0.0357,0.1920,163.9,68.5,37.0,-166.0,-0.758535+0.218940j,1.962981+ 4.983316j,0.028511+0.021485j,-0.186297-0.046449j,0.202581-0.189810j,0.277609,1.090147,1.509380,0.336487,-0.729611+0.174169j,0.008925-0.146073j,-0.870981-0.207916j,0.035522+0.581403j,-0.870981+0.207916j,0.035522-0.581403j,2.795898+5.867048j,26.048096-45.843039j,2.795898,5.867048,26.048096,-45.843039,15.107596,7.199412,1.004848e-11,27.484173,36.088846,28.410089,2.546390e-12,19.931241
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,BFP760_w_noise_VCE_2.0V_IC_35mA.s2p,BFP760_w_noise_VCE_2.0V_IC_35mA,2.2,2.0,35.0,0.6601,10.813,0.0415,0.2986,176.2,74.2,33.2,-152.9,-0.658649+0.043747j,2.944166+10.404463j,0.034726+0.022724j,-0.265818-0.136026j,0.315223-0.350241j,0.471205,0.776776,1.124536,0.431396,-0.622499-0.092231j,-0.042874-0.352921j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
150,BFP760_VCE_3.0V_IC_70mA.s2p,BFP760_VCE_3.0V_IC_70mA,2.2,3.0,70.0,0.6913,9.889,0.0351,0.2313,168.5,71.6,41.9,-153.8,-0.677422+0.137823j,3.121453+ 9.383435j,0.026125+0.023441j,-0.207536-0.102120j,0.293071-0.277740j,0.403770,0.909864,1.261366,0.412574,-0.644962+0.050254j,0.029276-0.249875j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
151,BFP760_VCE_3.5V_IC_22mA.s2p,BFP760_VCE_3.5V_IC_22mA,2.2,3.5,22.0,0.6482,10.919,0.0466,0.3025,-175.2,77.4,27.4,-136.5,-0.645927-0.054240j,2.381906+10.656035j,0.041372+0.021445j,-0.219426-0.208227j,0.260416-0.345543j,0.432685,0.663830,1.141440,0.484126,-0.660736-0.184287j,-0.069958-0.445548j,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN+ NaNj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
152,BFP760_VCE_3.5V_IC_50mA.s2p,BFP760_VCE_3.5V_IC_50mA,2.2,3.5,50.0,0.6555,11.244,0.0378,0.2663,173.5,73.7,38.4,-151.4,-0.651286+

In [ ]:
filtered_df = df_s_parameters[(df_s_parameters['delta'] < 1) & (df_s_parameters['K'] > 1)]

print("Filas donde delta < 1 y K > 1:")
display(filtered_df)

Filas donde delta < 1 y K > 1:


,filename,network_name,frequency_ghz,VCE,IC,s11_mag,s21_mag,s12_mag,s22_mag,s11_deg,s21_deg,s12_deg,s22_deg,s11_complex,s21_complex,s12_complex,s22_complex,delta,delta_modulo,K,B1,B2,C1,C2,ro_s(m),ro_L(m),ro_in,ro_out,Z_in,Z_out,R_in,X_in,R_out,X_out,R_in_p,X_in_p,C_in,Z_0_QWin,Z_0_QWout,X_L_p,C_out,Gt_max_db
4,BFP760_VCE_1.5V_IC_60mA.s2p,BFP760_VCE_1.5V_IC_60mA,2.2,1.5,60.0,0.7895,5.356,0.0357,0.1920,163.9,68.5,37.0,-166.0,-0.758535+0.218940j,1.962981+4.983316j,0.028511+0.021485j,-0.186297-0.046449j,0.202581-0.189810j,0.277609,1.090147,1.509380,0.336487,-0.729611+0.174169j,0.008925-0.146073j,-0.870981-0.207916j,0.035522+0.581403j,-0.870981+0.207916j,0.035522-0.581403j,2.795898+5.867048j,26.048096-45.843039j,2.795898,5.867048,26.048096,-45.843039,15.107596,7.199412,1.004848e-11,27.484173,36.088846,28.410089,2.546390e-12,19.931241
7,BFP760_VCE_1.0V_IC_55mA.s2p,BFP760_VCE_1.0V_IC_55mA,2.2,1.0,55.0,0.8468,2.617,0.0388,0.2446,162.1,67.2,33.5,172.7,-0.805810+0.260270j,1.014128+2.412516j,0.032355+0.021415j,-0.242617+0.031080j,0.206267-0.187965j,0.279064,1.482069,1.579364,0.264882,-0.749924+0.221077j,-0.027484-0.066699j,-0.832553-0.245436j,-0.112865+0.273905j,-0.832553+0.245436j,-0.112865-0.273905j,3.607091+7.179655j,34.725642-20.853141j,3.607091,7.179655,34.725642,-20.853141,17.897679,8.991874,8.045393e-12,29.914611,41.668718,83.262379,8.688577e-13,14.180383
21,BFP760_VCE_2.0V_IC_70mA.s2p,BFP760_VCE_2.0V_IC_70mA,2.2,2.0,70.0,0.7915,5.262,0.0337,0.1558,163.7,68.9,39.5,-155.5,-0.759686+0.222148j,1.894303+4.909202j,0.026004+0.021436j,-0.141772-0.064609j,0.178029-0.150675j,0.233232,1.138140,1.547801,0.343404,-0.744181+0.189284j,0.026946-0.139526j,-0.855111-0.217499j,0.100517+0.520474j,-0.855111+0.217499j,0.100517-0.520474j,3.174194+6.234310j,33.288359-48.193671j,3.174194,6.234310,33.288359,-48.193671,15.418758,7.850448,9.215163e-12,27.765769,40.797279,34.536027,2.094716e-12,19.677949
24,BFP760_VCE_1.0V_IC_50mA.s2p,BFP760_VCE_1.0V_IC_50mA,2.2,1.0,50.0,0.8110,4.290,0.0394,0.2570,163.2,67.7,32.5,179.1,-0.776386+0.234405j,1.627867+3.969150j,0.033230+0.021170j,-0.256968+0.004037j,0.228492-0.229723j,0.324009,1.127671,1.486690,0.303346,-0.716743+0.176296j,-0.025722-0.120758j,-0.862050-0.212036j,-0.107278+0.503642j,-0.862050+0.212036j,-0.107278-0.503642j,3.016777+6.037152j,24.830247-34.036340j,3.016777,6.037152,24.830247,-34.036340,15.098280,7.544641,9.588680e-12,27.475698,35.235101,36.476083,1.983304e-12,18.197760
30,BFP760_VCE_1.0V_IC_60mA.s2p,BFP760_VCE_1.0V_IC_60mA,2.2,1.0,60.0,0.8655,1.886,0.0379,0.2516,160.7,65.9,35.2,169.0,-0.816860+0.286060j,0.770111+1.721605j,0.030970+0.021847j,-0.246977+0.048008j,0.201774-0.180008j,0.270399,1.823763,1.612672,0.241097,-0.758384+0.251289j,-0.030663-0.041314j,-0.828515-0.274527j,-0.133570+0.179967j,-0.828515+0.274527j,-0.133570-0.179967j,3.483608+8.029837j,36.048041-13.661082j,3.483608,8.029837,36.048041,-13.661082,21.992658,9.541141,7.582233e-12,33.160713,42.454706,131.936990,5.483159e-13,11.719975
62,BFP760_VCE_1.0V_IC_65mA.s2p,BFP760_VCE_1.0V_IC_65mA,2.2,1.0,65.0,0.8756,1.627,0.0373,0.2604,159.3,64.3,37.0,167.9,-0.819075+0.309503j,0.705563+1.466052j,0.029789+0.022448j,-0.254615+0.054585j,0.203546-0.183024j,0.273731,1.981022,1.623939,0.226204,-0.757259+0.274013j,-0.031249-0.032327j,-0.826927-0.299222j,-0.144083+0.149054j,-0.826927+0.299222j,-0.144083-0.149054j,3.306758+8.730808j,35.947398-11.197437j,3.306758,8.730808,35.947398,-11.197437,26.358644,9.983228,7.246469e-12,36.303336,42.395400,160.516187,4.506907e-13,10.725200
107,BFP760_VCE_1.0V_IC_70mA.s2p,BFP760_VCE_1.0V_IC_70mA,2.2,1.0,70.0,0.8826,1.507,0.0369,0.2691,158.1,62.9,38.0,167.7,-0.818908+0.329199j,0.686506+1.341551j,0.029078+0.022718j,-0.262923+0.057326j,0.206953-0.188104j,0.279666,2.039401,1.628355,0.215219,-0.753712+0.291606j,-0.031524-0.028585j,-0.825519-0.319388j,-0.152695+0.138460j,-0.825519+0.319388j,-0.152695-0.138460j,3.151953+9.299315j,35.519317-10.272453j,3.151953,9.299315,35.519317,-10.272453,30.588044,10.367652,6.977776e-12,39.107